# 面试问题：ML Data Drift 怎样同时监控数值分布、类别占比和缺失率，而不等标签到齐？

        ## 可直接复述的回答主线

        1. 只看线上准确率需要等待标签，标签延迟期间即使输入分布已经变化也不会报警。
2. 数值特征可以同时计算 PSI 的分箱贡献和手写 KS 最大经验分布差，避免只看均值。
3. 类别特征用 Total Variation 比较占比，新类别会直接贡献分布距离。
4. 缺失率要独立监控，因为先过滤 None 再算 KS 会把采集故障隐藏掉。
5. 多指标门禁应逐窗口打印 PSI、KS、TV、missing delta、触发原因和已知标签状态。
6. 生产还需基线版本、季节性分层、样本量置信区间、多重检验、告警抑制、根因定位和再训练门禁。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是反欺诈模型的一个历史参考集和八个日窗口，每个窗口十笔交易，字段包括 amount、device_score、channel 与延迟到达的 accuracy。第 5 天金额整体升高，第 6 天设备分数大面积缺失，第 7 天出现 affiliate 渠道。

In [1]:
import math  # 计算 PSI 的对数分项并检查有限数值。
reference = {"amount": [20, 25, 35, 40, 55, 60, 75, 80, 95, 100], "device_score": [0.10, 0.18, 0.27, 0.35, 0.44, 0.53, 0.62, 0.71, 0.83, 0.94], "channel": ["web", "web", "web", "web", "app", "app", "app", "app", "store", "store"]}  # 定义十笔历史参考交易分布。
windows = [{"id": "day-01", "amount": [21, 26, 34, 42, 54, 61, 74, 82, 94, 101], "device_score": [0.11, 0.19, 0.26, 0.36, 0.43, 0.54, 0.63, 0.72, 0.82, 0.93], "channel": ["web", "web", "web", "web", "app", "app", "app", "app", "store", "store"], "label_accuracy": 0.91, "expected_alert": False}, {"id": "day-02", "amount": [19, 28, 33, 45, 52, 66, 73, 88, 92, 104], "device_score": [0.09, 0.21, 0.29, 0.34, 0.46, 0.51, 0.65, 0.69, 0.85, 0.91], "channel": ["app", "web", "web", "web", "app", "app", "web", "app", "store", "store"], "label_accuracy": 0.90, "expected_alert": False}, {"id": "day-03", "amount": [22, 24, 37, 43, 58, 63, 78, 84, 97, 99], "device_score": [0.12, 0.17, 0.28, 0.38, 0.42, 0.56, 0.60, 0.74, 0.80, 0.95], "channel": ["web", "app", "web", "web", "app", "app", "web", "app", "store", "store"], "label_accuracy": 0.89, "expected_alert": False}, {"id": "day-04", "amount": [18, 29, 31, 48, 51, 69, 71, 89, 91, 108], "device_score": [0.08, 0.22, 0.25, 0.39, 0.41, 0.57, 0.59, 0.76, 0.79, 0.96], "channel": ["web", "web", "app", "web", "app", "web", "app", "app", "store", "store"], "label_accuracy": None, "expected_alert": False}, {"id": "day-05", "amount": [112, 121, 130, 138, 149, 157, 166, 178, 190, 205], "device_score": [0.10, 0.18, 0.27, 0.35, 0.44, 0.53, 0.62, 0.71, 0.83, 0.94], "channel": ["web", "web", "web", "web", "app", "app", "app", "app", "store", "store"], "label_accuracy": None, "expected_alert": True}, {"id": "day-06", "amount": [20, 25, 35, 40, 55, 60, 75, 80, 95, 100], "device_score": [None, None, None, None, None, None, 0.25, 0.45, 0.65, 0.85], "channel": ["web", "web", "web", "web", "app", "app", "app", "app", "store", "store"], "label_accuracy": None, "expected_alert": True}, {"id": "day-07", "amount": [20, 25, 35, 40, 55, 60, 75, 80, 95, 100], "device_score": [0.10, 0.18, 0.27, 0.35, 0.44, 0.53, 0.62, 0.71, 0.83, 0.94], "channel": ["affiliate", "affiliate", "affiliate", "affiliate", "affiliate", "affiliate", "affiliate", "affiliate", "web", "web"], "label_accuracy": None, "expected_alert": True}, {"id": "day-08", "amount": [23, 27, 32, 47, 53, 67, 72, 86, 93, 102], "device_score": [0.13, 0.16, 0.30, 0.33, 0.48, 0.50, 0.67, 0.68, 0.87, 0.90], "channel": ["web", "web", "web", "app", "app", "web", "app", "app", "store", "store"], "label_accuracy": 0.90, "expected_alert": False}]  # 定义八个稳定、漂移、缺失和新类别窗口。
print("教学实验输入：反欺诈特征日窗口")  # 标记下方为脱敏离线监控数据。
print("window   amount范围/均值       device缺失  channel计数                         label_accuracy  expected")  # 输出窗口预览表头。
for window in windows:  # 逐天展示业务特征和标签可用性。
    channel_counts = {channel: window["channel"].count(channel) for channel in sorted(set(window["channel"]))}  # 汇总当前渠道占比。
    mean_amount = sum(window["amount"]) / len(window["amount"])  # 计算当前金额均值供直观预览。
    missing = sum(value is None for value in window["device_score"])  # 统计设备分数缺失数。
    print(f"{window['id']:<8} [{min(window['amount']):>3},{max(window['amount']):>3}]/{mean_amount:>6.1f} {missing:>10}/10  {str(channel_counts):<36} {str(window['label_accuracy']):>14} {str(window['expected_alert']):>9}")  # 输出当前窗口的真实监控字段。

教学实验输入：反欺诈特征日窗口
window   amount范围/均值       device缺失  channel计数                         label_accuracy  expected
day-01   [ 21,101]/  58.9          0/10  {'app': 4, 'store': 2, 'web': 4}               0.91     False
day-02   [ 19,104]/  60.0          0/10  {'app': 4, 'store': 2, 'web': 4}                0.9     False
day-03   [ 22, 99]/  60.5          0/10  {'app': 4, 'store': 2, 'web': 4}               0.89     False
day-04   [ 18,108]/  60.5          0/10  {'app': 4, 'store': 2, 'web': 4}               None     False
day-05   [112,205]/ 154.6          0/10  {'app': 4, 'store': 2, 'web': 4}               None      True
day-06   [ 20,100]/  58.5          6/10  {'app': 4, 'store': 2, 'web': 4}               None      True
day-07   [ 20,100]/  58.5          0/10  {'affiliate': 8, 'web': 2}                     None      True
day-08   [ 23,102]/  60.2          0/10  {'app': 4, 'store': 2, 'web': 4}                0.9     False


## 2. Baseline / 基线：只在标签到齐后看 accuracy

基线阈值为 0.80。第 5–7 天标签尚未回流，因此不会报警；它完全看不到金额、缺失率和新渠道已经变化。

In [2]:
baseline_rows = []  # 保存八个窗口的标签延迟基线决策。
for window in windows:  # 逐天检查是否有可用准确率。
    accuracy = window["label_accuracy"]  # 读取延迟监督指标。
    alert = accuracy is not None and accuracy < 0.80  # 只有标签到齐且准确率低才告警。
    reason = "low_accuracy" if alert else "labels_pending" if accuracy is None else "accuracy_ok"  # 解释基线为什么未报警。
    baseline_rows.append({"id": window["id"], "alert": alert, "reason": reason, "correct": alert == window["expected_alert"]})  # 保存决策和期望对照。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(baseline_rows)  # 计算漂移窗口决策准确率。
print("Baseline 标签准确率监控")  # 标记下表展示标签延迟盲区。
print("window   label_accuracy  alert  expected  correct  reason")  # 输出基线结果表头。
for row, window in zip(baseline_rows, windows):  # 逐天展示延迟标签决策。
    print(f"{row['id']:<8} {str(window['label_accuracy']):>14} {str(row['alert']):>6} {str(window['expected_alert']):>9} {str(row['correct']):>8}  {row['reason']}")  # 输出当前窗口的告警结果。

Baseline 标签准确率监控
window   label_accuracy  alert  expected  correct  reason
day-01             0.91  False     False     True  accuracy_ok
day-02              0.9  False     False     True  accuracy_ok
day-03             0.89  False     False     True  accuracy_ok
day-04             None  False     False     True  labels_pending
day-05             None  False      True    False  labels_pending
day-06             None  False      True    False  labels_pending
day-07             None  False      True    False  labels_pending
day-08              0.9  False     False     True  accuracy_ok


## 3. 底层实现：PSI 分项、经验 KS、类别 TV 与缺失率

金额使用固定参考分箱计算每箱 `(actual-reference)*log(actual/reference)`；KS 枚举两组样本的经验 CDF；TV 对类别概率差取一半 L1。

In [3]:
amount_edges = [float("-inf"), 30.0, 50.0, 70.0, 90.0, float("inf")]  # 用历史业务区间固定金额 PSI 分箱。
def binned_proportions(values, edges, epsilon=1.0e-4):  # 计算带平滑的分箱概率。
    counts = [0 for _ in range(len(edges) - 1)]  # 初始化每个区间计数。
    for value in values:  # 逐数值定位所属左闭右开区间。
        index = next(index for index in range(len(edges) - 1) if edges[index] <= value < edges[index + 1])  # 找到当前金额分箱。
        counts[index] += 1  # 累加当前分箱样本数。
    denominator = len(values) + epsilon * len(counts)  # 加入对称平滑后的归一化分母。
    return [(count + epsilon) / denominator for count in counts], counts  # 返回概率和原始计数。
def population_stability_index(reference_values, actual_values, edges):  # 手写 PSI 总值和逐箱贡献。
    reference_probabilities, reference_counts = binned_proportions(reference_values, edges)  # 计算参考分布。
    actual_probabilities, actual_counts = binned_proportions(actual_values, edges)  # 计算当前分布。
    contributions = []  # 保存每个分箱的 PSI 中间量。
    for index, (reference_probability, actual_probability) in enumerate(zip(reference_probabilities, actual_probabilities)):  # 逐箱计算稳定性贡献。
        contribution = (actual_probability - reference_probability) * math.log(actual_probability / reference_probability)  # 应用 PSI 标准公式。
        contributions.append({"bin": (edges[index], edges[index + 1]), "reference_count": reference_counts[index], "actual_count": actual_counts[index], "reference_p": reference_probability, "actual_p": actual_probability, "contribution": contribution})  # 保存当前箱计数、概率和贡献。
    return sum(row["contribution"] for row in contributions), contributions  # 返回 PSI 总和与分项。
def empirical_ks(reference_values, actual_values):  # 手写两个数值样本的 KS 最大 CDF 差。
    reference_clean = sorted(value for value in reference_values if value is not None)  # 过滤参考缺失并排序。
    actual_clean = sorted(value for value in actual_values if value is not None)  # 过滤当前缺失并排序。
    points = sorted(set(reference_clean + actual_clean))  # 枚举所有经验 CDF 可能变化的位置。
    differences = []  # 保存每个位置的两个 CDF 与差值。
    for point in points:  # 逐候选阈值计算经验分布。
        reference_cdf = sum(value <= point for value in reference_clean) / len(reference_clean)  # 计算参考经验 CDF。
        actual_cdf = sum(value <= point for value in actual_clean) / len(actual_clean) if actual_clean else 0.0  # 计算当前经验 CDF并保护全缺失。
        differences.append({"point": point, "reference_cdf": reference_cdf, "actual_cdf": actual_cdf, "difference": abs(reference_cdf - actual_cdf)})  # 保存当前位置差异。
    maximum = max(differences, key=lambda row: row["difference"]) if differences else {"point": None, "reference_cdf": 0.0, "actual_cdf": 0.0, "difference": 1.0}  # 找到 KS 最大差位置。
    return maximum["difference"], maximum  # 返回 KS 统计量和最强证据。
def categorical_tv(reference_values, actual_values):  # 手写类别分布 Total Variation 距离。
    categories = sorted(set(reference_values) | set(actual_values))  # 合并参考和当前全部类别，包括新类别。
    rows = []  # 保存每个类别概率差。
    for category in categories:  # 逐类别计算两期占比。
        reference_probability = reference_values.count(category) / len(reference_values)  # 计算参考类别占比。
        actual_probability = actual_values.count(category) / len(actual_values)  # 计算当前类别占比。
        rows.append({"category": category, "reference_p": reference_probability, "actual_p": actual_probability, "absolute_difference": abs(actual_probability - reference_probability)})  # 保存类别分项。
    return 0.5 * sum(row["absolute_difference"] for row in rows), rows  # 返回一半 L1 距离和分项。
day_five_psi, day_five_bins = population_stability_index(reference["amount"], windows[4]["amount"], amount_edges)  # 计算金额整体升高日的 PSI 分项。
print("day-05金额 PSI 分箱中间量")  # 标记下表展示不是只输出最终布尔值。
for row in day_five_bins:  # 逐分箱展示参考和当前差异。
    print(row)  # 输出当前箱边界、计数、概率与 PSI 贡献。

day-05金额 PSI 分箱中间量
{'bin': (-inf, 30.0), 'reference_count': 2, 'actual_count': 0, 'reference_p': 0.2, 'actual_p': 9.99950002499875e-06, 'contribution': 1.9806084798332424}
{'bin': (30.0, 50.0), 'reference_count': 2, 'actual_count': 0, 'reference_p': 0.2, 'actual_p': 9.99950002499875e-06, 'contribution': 1.9806084798332424}
{'bin': (50.0, 70.0), 'reference_count': 2, 'actual_count': 0, 'reference_p': 0.2, 'actual_p': 9.99950002499875e-06, 'contribution': 1.9806084798332424}
{'bin': (70.0, 90.0), 'reference_count': 2, 'actual_count': 0, 'reference_p': 0.2, 'actual_p': 9.99950002499875e-06, 'contribution': 1.9806084798332424}
{'bin': (90.0, inf), 'reference_count': 2, 'actual_count': 10, 'reference_p': 0.2, 'actual_p': 0.9999600019998999, 'contribution': 1.2874539582093365}


## 4. 逐窗口结果与结果解读

阈值分别为 amount PSI>0.25、device KS>0.35、缺失率增量>0.30、channel TV>0.35。任一指标触发就报警，并保留具体原因。

In [4]:
corrected_rows = []  # 保存八个窗口的多指标漂移结果。
reference_missing_rate = sum(value is None for value in reference["device_score"]) / len(reference["device_score"])  # 计算参考设备分数缺失率。
for window in windows:  # 逐窗口计算四类输入漂移指标。
    psi_value, psi_bins = population_stability_index(reference["amount"], window["amount"], amount_edges)  # 计算金额 PSI。
    ks_value, ks_point = empirical_ks(reference["device_score"], window["device_score"])  # 计算非缺失设备分数 KS。
    missing_rate = sum(value is None for value in window["device_score"]) / len(window["device_score"])  # 计算当前设备缺失率。
    missing_delta = missing_rate - reference_missing_rate  # 计算相对参考缺失率增量。
    tv_value, tv_rows = categorical_tv(reference["channel"], window["channel"])  # 计算渠道类别 TV。
    reasons = []  # 保存所有独立触发原因。
    if psi_value > 0.25:  # 检查金额分布稳定性。
        reasons.append("amount_psi")  # 记录金额 PSI 漂移。
    if ks_value > 0.35:  # 检查设备分数经验 CDF 差。
        reasons.append("device_ks")  # 记录设备分数漂移。
    if missing_delta > 0.30:  # 检查设备字段采集缺失。
        reasons.append("device_missing")  # 记录缺失率异常。
    if tv_value > 0.35:  # 检查渠道占比或新类别变化。
        reasons.append("channel_tv")  # 记录类别漂移。
    alert = bool(reasons)  # 任一输入漂移门禁触发即报警。
    corrected_rows.append({"id": window["id"], "psi": psi_value, "ks": ks_value, "ks_point": ks_point, "missing_delta": missing_delta, "tv": tv_value, "alert": alert, "reasons": reasons, "correct": alert == window["expected_alert"]})  # 保存指标、证据和期望对照。
corrected_accuracy = sum(row["correct"] for row in corrected_rows) / len(corrected_rows)  # 计算多指标漂移决策准确率。
print("window   PSI      KS    missing_delta   TV     baseline/monitor  expected  reasons")  # 输出逐窗口同数据对照表头。
for baseline, corrected, window in zip(baseline_rows, corrected_rows, windows):  # 逐天比较标签基线和输入监控。
    print(f"{corrected['id']:<8} {corrected['psi']:>6.3f} {corrected['ks']:>7.3f} {corrected['missing_delta']:>14.2f} {corrected['tv']:>6.3f} {str(baseline['alert']):>5}/{str(corrected['alert']):<7} {str(window['expected_alert']):>8}  {corrected['reasons']}")  # 输出当前窗口的全部漂移信号。
print(f"结果解读：标签延迟基线决策准确率={baseline_accuracy:.1%}，多指标输入监控={corrected_accuracy:.1%}；day-05/06/07分别定位金额、缺失和新渠道。")  # 解释不同指标负责的真实故障类型。

window   PSI      KS    missing_delta   TV     baseline/monitor  expected  reasons
day-01    0.000   0.100           0.00  0.000 False/False      False  []
day-02    0.000   0.100           0.00  0.000 False/False      False  []
day-03    0.000   0.100           0.00  0.000 False/False      False  []
day-04    0.000   0.100           0.00  0.000 False/False      False  []
day-05    9.210   0.000           0.00  0.000 False/True        True  ['amount_psi']
day-06    0.000   0.250           0.60  0.000 False/True        True  ['device_missing']
day-07    0.000   0.000           0.00  0.800 False/True        True  ['channel_tv']
day-08    0.000   0.100           0.00  0.000 False/False      False  []
结果解读：标签延迟基线决策准确率=62.5%，多指标输入监控=100.0%；day-05/06/07分别定位金额、缺失和新渠道。


## 5. 失败案例与修正：均值完全相同但分布已经双峰化

构造五笔 10 元和五笔 107 元交易，均值仍为历史 58.5 元。均值监控认为无变化；PSI/KS 会看到中间金额消失和两端堆积。

In [5]:
mean_matched_bimodal = [10, 10, 10, 10, 10, 107, 107, 107, 107, 107]  # 构造均值与参考相同但形状不同的十笔交易。
reference_mean = sum(reference["amount"]) / len(reference["amount"])  # 计算历史金额均值。
bimodal_mean = sum(mean_matched_bimodal) / len(mean_matched_bimodal)  # 计算双峰窗口均值。
mean_delta = abs(bimodal_mean - reference_mean)  # 模拟只看均值的漂移统计量。
bimodal_psi, bimodal_bins = population_stability_index(reference["amount"], mean_matched_bimodal, amount_edges)  # 计算双峰分布 PSI。
bimodal_ks, bimodal_ks_point = empirical_ks(reference["amount"], mean_matched_bimodal)  # 计算双峰分布 KS 最大差。
mean_monitor_alert = mean_delta > 5.0  # 使用五元均值阈值的错误监控。
distribution_monitor_alert = bimodal_psi > 0.25 or bimodal_ks > 0.35  # 使用完整分布门禁检测双峰化。
print(f"错误行为：reference_mean={reference_mean:.1f}，bimodal_mean={bimodal_mean:.1f}，delta={mean_delta:.1f}，alert={mean_monitor_alert}")  # 展示相同均值隐藏结构漂移。
print(f"修正行为：PSI={bimodal_psi:.4f}，KS={bimodal_ks:.4f}@{bimodal_ks_point['point']}，alert={distribution_monitor_alert}")  # 展示分布统计量捕获失败案例。

错误行为：reference_mean=58.5，bimodal_mean=58.5，delta=0.0，alert=False
修正行为：PSI=6.4916，KS=0.5000@10，alert=True


## 6. 生产边界

十条样本的统计量方差很大，不能照搬阈值。生产要做最小样本门禁、置信区间、季节/地区/渠道分层、参考版本与数据契约、多个特征的 FDR 控制、告警连续窗口确认、特征 lineage、标签性能回填和再训练审批。

In [6]:
drift_diagnostics = {"windows": len(windows), "records_per_window": len(windows[0]["amount"]), "baseline_accuracy": baseline_accuracy, "multimetric_accuracy": corrected_accuracy, "alerts": sum(row["alert"] for row in corrected_rows), "pending_label_windows": sum(window["label_accuracy"] is None for window in windows), "mean_blind_spot_detected": distribution_monitor_alert and not mean_monitor_alert}  # 汇总窗口、标签延迟和分布检测指标。
print("生产监控快照：", drift_diagnostics)  # 输出漂移平台应持续观察的信号。

生产监控快照： {'windows': 8, 'records_per_window': 10, 'baseline_accuracy': 0.625, 'multimetric_accuracy': 1.0, 'alerts': 3, 'pending_label_windows': 4, 'mean_blind_spot_detected': True}


## 7. 最小回归测试

断言覆盖数据规模、稳定窗口、三类真实漂移、标签延迟盲区和均值反例。

In [7]:
assert len(windows) >= 6 and all(len(window["amount"]) >= 6 for window in windows)  # 保证至少六个窗口且每窗有足够真实记录。
assert corrected_accuracy > baseline_accuracy and corrected_accuracy == 1.0  # 保证多指标监控在同一窗口上优于标签延迟基线。
assert not any(row["alert"] for row in corrected_rows[:4]) and not corrected_rows[7]["alert"]  # 保证稳定和恢复窗口不会误报。
assert "amount_psi" in corrected_rows[4]["reasons"] and "device_missing" in corrected_rows[5]["reasons"] and "channel_tv" in corrected_rows[6]["reasons"]  # 保证三类故障被对应指标定位。
assert all(not baseline_rows[index]["alert"] for index in (4, 5, 6))  # 保证标签未到时的监控盲区真实复现。
assert mean_delta < 1.0e-12 and not mean_monitor_alert and distribution_monitor_alert  # 保证均值相同的双峰反例被分布指标修正。